## load_silver_noaa_climate
Conforms `bronze.climate_normals_stations` (station grain) into `silver.fact_noaa_climate_cbsa` (CBSA grain, geo-keyed, **no date** — a static climate characteristic, joined to the housing facts on `geo_key`).

**Rollup:** trim + cast the 13 measures→DOUBLE (Bronze keeps NCEI leading-space padding); station cast failures → `silver.quarantine`. **INNER-join** the station→CBSA crosswalk — rural stations outside every CBSA drop (expected, *not* quarantine). Group by CBSA: **mean-of-stations** (`F.avg`) per measure — `avg` ignores nulls, so precip-only stations contribute only their populated measures. Resolve `geo_key` via `dim_geo`; MERGE on `geo_key`. StepLog + transform_detail_log. Design: `weather_silver_gold_design.md` §3.

In [ ]:
%run "../libs/notebook_init"

In [ ]:
# notebook_init injects: BRONZE, SILVER, CROSSWALKS, AUDIT, PIPELINE_RUN_ID, STATUS_*, StepLog,
# Utils, transform_detail_log_insert, spark, dbutils, F, datetime, timezone.

STEP_SEQUENCE = 2                                  # position owned by the orchestrator
SOURCE_SYSTEM = "climate_normals"
SOURCE_TABLE  = f"{BRONZE}.climate_normals_stations"
TARGET_TABLE  = f"{SILVER}.fact_noaa_climate_cbsa"
QUARANTINE    = f"{SILVER}.quarantine"
DIM_GEO       = f"{SILVER}.dim_geo"
STATION_XWALK = f"{CROSSWALKS}station_to_cbsa.csv"  # station -> cbsa_code (committed reference)

# The 13 climate-normal measures, rolled up station -> CBSA by MEAN-OF-STATIONS. Same list as
# the bronze.climate_normals_stations measures and the fact_noaa_climate_cbsa DDL columns.
MEASURES = [
    "ann_tavg_normal", "djf_tavg_normal", "mam_tavg_normal", "jja_tavg_normal", "son_tavg_normal",
    "ann_tmax_normal", "ann_tmin_normal", "jja_tmax_normal", "djf_tmin_normal",
    "ann_prcp_normal", "ann_snow_normal", "ann_htdd_normal", "ann_cldd_normal",
]

In [ ]:
# Open the pipeline_step_log row (RUNNING). Closed explicitly by step.succeed() in the write
# cell, or by step.fail(e) in any work cell's 2-line handler.
nb = Utils.get_notebook_context(dbutils)
step = StepLog(
    spark, AUDIT, dbutils,
    pipeline_run_id = PIPELINE_RUN_ID,
    step_sequence   = STEP_SEQUENCE,
    notebook_folder = nb["notebook_folder"],
    notebook_name   = nb["notebook_name"],
    layer           = "silver",
    target_table    = TARGET_TABLE,
)
print(f"load_silver_noaa_climate: step_log_id={step.step_log_id}")

In [ ]:
# Read Normals stations (all-STRING), trim the NCEI leading-space padding, cast the 13 measures.
# cast_err (CLAUDE.md §11): non-null + non-blank + cast-is-null = a genuine failure; a blank is
# "not reported" -> null (precip-only stations legitimately have null temp measures), not an
# error. raw_payload keeps the full Bronze row for any quarantined station.
try:
    bronze = spark.table(SOURCE_TABLE)
    rows_read = bronze.count()

    cast_err = lambda c: (F.col(c).isNotNull()) & (F.trim(F.col(c)) != F.lit("")) \
                         & (F.trim(F.col(c)).cast("double").isNull())
    typed = bronze.select(
        F.col("station"),
        *[F.trim(F.col(c)).cast("double").alias(c) for c in MEASURES],
        F.array_compact(F.array(*[F.when(cast_err(c), F.lit(c)) for c in MEASURES])).alias("cast_errors"),
        F.to_json(F.struct(*[F.col(x) for x in bronze.columns])).alias("raw_payload"),
    )
    step.rows_read = rows_read
    print(f"load_silver_noaa_climate: read {rows_read:,} Normals station rows")
except Exception as e:
    step.fail(e); raise

In [ ]:
# Quarantine station cast failures, roll the good stations up to CBSA (mean-of-stations), resolve
# geo_key, MERGE. §11.4 audit vars pre-declared. Quarantine idempotent: DELETE this source's rows
# first, then append. Rural stations (no CBSA in the crosswalk) drop on the INNER join — expected,
# not a quarantine case.
transform_started = datetime.now(timezone.utc)
rows_rejected = rows_inserted = rows_updated = 0
try:
    good = typed.where(F.size("cast_errors") == 0)
    bad  = typed.where(F.size("cast_errors") > 0)
    rows_rejected = bad.count()

    spark.sql(f"DELETE FROM {QUARANTINE} WHERE source_system = '{SOURCE_SYSTEM}'")
    if rows_rejected > 0:
        bad.select(
            F.expr("uuid()").alias("quarantine_id"),
            F.lit(SOURCE_SYSTEM).alias("source_system"),
            F.lit(None).cast("string").alias("source_file_path"),
            F.col("station").alias("natural_key"),
            F.col("raw_payload"),
            F.concat(F.lit("cast_failed:"), F.concat_ws(",", F.col("cast_errors"))).alias("quarantine_reason"),
            F.current_timestamp().alias("quarantined_ts"),
        ).write.format("delta").mode("append").saveAsTable(QUARANTINE)

    # Station -> CBSA crosswalk (committed reference). INNER join: rural stations drop (expected).
    xwalk  = spark.read.option("header", True).csv(STATION_XWALK)
    joined = good.join(xwalk, "station", "inner")

    # Mean-of-stations per measure. F.avg ignores nulls, so a precip-only station contributes to
    # ann_prcp_normal/ann_snow_normal only and is skipped for the temperature means.
    rolled = joined.groupBy("cbsa_code").agg(*[F.avg(c).alias(c) for c in MEASURES])

    # Resolve geo_key on cbsa_code. Null geo_key is unexpected (crosswalk CBSAFP + dim_geo are
    # both the same CBSA vintage); quarantine any such CBSA rather than drop it silently.
    geo = spark.table(DIM_GEO).select("geo_key", "cbsa_code")
    staged = rolled.join(geo, "cbsa_code", "left")
    matched   = staged.where(F.col("geo_key").isNotNull())
    unmatched = staged.where(F.col("geo_key").isNull())
    unmatched_n = unmatched.count()
    if unmatched_n > 0:
        unmatched.select(
            F.expr("uuid()").alias("quarantine_id"),
            F.lit(SOURCE_SYSTEM).alias("source_system"),
            F.lit(None).cast("string").alias("source_file_path"),
            F.col("cbsa_code").alias("natural_key"),
            F.to_json(F.struct(*rolled.columns)).alias("raw_payload"),
            F.lit("unmatched_geography").alias("quarantine_reason"),
            F.current_timestamp().alias("quarantined_ts"),
        ).write.format("delta").mode("append").saveAsTable(QUARANTINE)
        rows_rejected += unmatched_n

    fact_cols = ["geo_key"] + MEASURES
    matched.select(
        *[F.col(c) for c in fact_cols],
        F.current_timestamp().alias("inserted_ts"),
        F.current_timestamp().alias("updated_ts"),
    ).createOrReplaceTempView("noaa_climate_staging")

    set_clause = ", ".join(f"t.{c}=s.{c}" for c in MEASURES) + ", t.updated_ts=s.updated_ts"
    cols_csv   = ", ".join(fact_cols + ["inserted_ts", "updated_ts"])
    vals_csv   = ", ".join(f"s.{c}" for c in fact_cols + ["inserted_ts", "updated_ts"])
    metrics = spark.sql(f"""
        MERGE INTO {TARGET_TABLE} t USING noaa_climate_staging s
        ON t.geo_key = s.geo_key
        WHEN MATCHED THEN UPDATE SET {set_clause}
        WHEN NOT MATCHED THEN INSERT ({cols_csv}) VALUES ({vals_csv})
    """).first().asDict()
    rows_inserted = metrics.get("num_inserted_rows") or 0
    rows_updated  = metrics.get("num_updated_rows") or 0

    step.rows_written = rows_inserted
    transform_detail_log_insert(
        spark, AUDIT, PIPELINE_RUN_ID, step.step_log_id, SOURCE_TABLE, TARGET_TABLE,
        status=STATUS_SUCCEEDED, started_timestamp=transform_started, rows_read=step.rows_read,
        rows_written=rows_inserted, rows_inserted=rows_inserted, rows_updated=rows_updated,
        rows_rejected=rows_rejected, ended_timestamp=datetime.now(timezone.utc))
    step.succeed()
    print(f"load_silver_noaa_climate: inserted={rows_inserted:,} updated={rows_updated:,} "
          f"quarantined={rows_rejected:,} (read={step.rows_read:,})")
except Exception as e:
    transform_detail_log_insert(
        spark, AUDIT, PIPELINE_RUN_ID, step.step_log_id, SOURCE_TABLE, TARGET_TABLE,
        status=STATUS_FAILED, started_timestamp=transform_started, rows_read=step.rows_read,
        rows_rejected=rows_rejected, error_message=f"{type(e).__name__}: {e}",
        ended_timestamp=datetime.now(timezone.utc))
    step.fail(e); raise